In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# Data load
df = pd.read_csv(r'..\data\raw\dna-sequence-dataset\human.txt', sep='\t')

# K-mer function — DNA sequence কে words এ ভাগ করে
def get_kmers(sequence, k=6):
    return [sequence[i:i+k].lower() for i in range(len(sequence) - k + 1)]

# Test করো
sample = df['sequence'][0]
kmers = get_kmers(sample)
print("Original sequence length:", len(sample))
print("Total k-mers:", len(kmers))
print("First 5 k-mers:", kmers[:5])

Original sequence length: 207
Total k-mers: 202
First 5 k-mers: ['atgccc', 'tgcccc', 'gcccca', 'ccccaa', 'cccaac']


In [2]:
# সব sequence এ k-mer apply করো
df['kmers'] = df['sequence'].apply(lambda x: ' '.join(get_kmers(x)))

print("Sample k-mer sentence:")
print(df['kmers'][0][:100])

Sample k-mer sentence:
atgccc tgcccc gcccca ccccaa cccaac ccaact caacta aactaa actaaa ctaaat taaata aaatac aatact atacta ta


In [3]:
# CountVectorizer দিয়ে text → numbers
vectorizer = CountVectorizer(ngram_range=(4,4))
X = vectorizer.fit_transform(df['kmers'])
y = df['class'].values

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)
print("Classes:", np.unique(y))

Feature matrix shape: (4380, 232414)
Labels shape: (4380,)
Classes: [0 1 2 3 4 5 6]


In [4]:
import pickle, os

os.makedirs('../data/processed', exist_ok=True)

# Processed data save করো
np.save('../data/processed/X_features.npy', X.toarray())
np.save('../data/processed/y_labels.npy', y)

# Vectorizer save করো (পরে predict এ লাগবে)
with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("✅ Features saved successfully!")
print("X shape:", X.shape)

✅ Features saved successfully!
X shape: (4380, 232414)


In [5]:
CLASS_NAMES = {
    0: "G Protein Coupled Receptors",
    1: "Tyrosine Kinase",
    2: "Tyrosine Phosphatase",
    3: "Synthetase",
    4: "Synthase",
    5: "Ion Channel",
    6: "Transcription Factor"
}

for class_id in sorted(df['class'].unique()):
    sample_seq = df[df['class'] == class_id]['sequence'].iloc[0]
    print(f"Class {class_id} ({CLASS_NAMES[class_id]}):")
    print(sample_seq)
    print()

Class 0 (G Protein Coupled Receptors):
ATGAGGCCCGAGCGTCCCCGGCCGCGCGGCAGCGCCCCCGGCCCGATGGAGACCCCGCCGTGGGACCCAGCCCGCAACGACTCGCTGCCGCCCACGCTGACCCCGGCCGTGCCCCCCTACGTGAAGCTTGGCCTCACCGTCGTCTACACCGTGTTCTACGCGCTGCTCTTCGTGTTCATCTACGTGCAGCTCTGGCTGGTGCTGCGTTACCGCCACAAGCGGCTCAGCTACCAGAGCGTCTTCCTCTTTCTCTGCCTCTTCTGGGCCTCCCTGCGGACCGTCCTCTTCTCCTTCTACTTCAAAGACTTCGTGGCGGCCAATTCGCTCAGCCCCTTCGTCTTCTGGCTGCTCTACTGCTTCCCTGTGTGCCTGCAGTTTTTCACCCTCACGCTGATGAACTTGTACTTCACGCAGGTTGCCCCTCTACCTGGCCTCCCTCTTCATCAGCCTTGTTTTCCTGTTGGTGAATTTAACCTGTGCTGTGCTGGTAAAGACGGGAAATTGGGAGAGGAAGGTTATCGTCTCTGTGCGAGTGGCCATTAA

Class 1 (Tyrosine Kinase):
ATGAAGACCATTACCGCCACTGGCGTCCTGTTTGTGCGGCTGGGTCCAACGCACAGCCCAAATCATAACTTTCAGGATGATTACCACGAGGATGGGTTCTGCCAGCCTTACCGGGGAATTGCCTGTGCACGCTTCATTGGCAACCGGACCATTTATGTGGACTCGCTTCAGATGCAGGGGGAGATTGAAAACCGAATCACAGCGGCCTTCACCATGATCGGCACGTCTACGCACCTGTCGGACCAGTGCTCACAGTTCGCCATCCCATCCTTCTGCCACTTCGTGTTTCCTCTGTGCGACGCGCGCTCCCGGACACCCAAGCCGCGTGAGCTGTGCCGCGACGAGTGCGAGGTGCTGGAGAGCGACCTGTGCCGCCAGGAGTACACCAT